<a href="https://colab.research.google.com/github/RafaXRR/ProgramacionAvanzada/blob/main/Mi_Copia_de_Sistemas_Operativos_U1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💻 Sistemas Operativos - Simulación de Cambio de Contexto y Excepciones de Hardware (Modo Dual / BCP)
**Asignatura:** Fundamentos y Diseño de Sistemas Operativos (6.º Semestre)    
**Unidad I:** Conceptos y Evolución de los Sistemas Operativos  
**Tiempo estimado:** 2 horas (Asíncrono)  
**Repositorio Base:** Integración con GitHub  

---

## 1. Fundamentación Teórica del Modo Dual y Excepciones

### 1. Niveles de Ejecución (Modo Dual)
Para garantizar la estabilidad y la seguridad del sistema informático, la arquitectura del procesador proporciona dos o más niveles de privilegio:
* **Modo Usuario (Nivel no privilegiado):** Las aplicaciones de usuario se ejecutan de manera restringida. Tienen prohibido ejecutar instrucciones máquina sensibles (como alterar puertos de E/S o modificar registros de gestión de memoria) y no pueden acceder al espacio de direcciones del sistema operativo.
* **Modo Núcleo / Kernel (Nivel privilegiado):** El monitor residente o kernel del sistema operativo cuenta con privilegios absolutos sobre el hardware, pudiendo ejecutar la totalidad del juego de instrucciones máquina y manipular todos los registros de control.

El nivel de ejecución activo está determinado por un bit especial dentro del **Registro de Estado** o **PSW** (*Program Status Word*).

### 2. Mecanismo de Interrupción y Trap de Excepción
Cuando un proceso en modo usuario intenta ejecutar una instrucción restringida o comete una falla aritmética (como una división entre cero), el hardware del procesador reacciona inmediatamente:
1. Cancela el ciclo de instrucción lineal actual.
2. Salva automáticamente los registros críticos **Contador de Programa (PC)** y **Puntero de Pila (SP)** en la pila del sistema o en la estructura de control del proceso.
3. Modifica la **PSW**, alternando forzosamente el nivel de ejecución a **Modo Núcleo**.
4. Salta a la dirección física apuntada por el vector de interrupciones para ejecutar la **Rutina de Tratamiento de Interrupción (ISR)** del kernel.

### 3. El Bloque de Control de Proceso (BCP)
El **BCP** (*Process Control Block*) es la estructura de datos clave donde el sistema operativo resguarda la imagen y el contexto de ejecución de un proceso. Durante un cambio de contexto o una excepción de hardware, el kernel actualiza el BCP con los valores exactos de los registros físicos (PC, SP, acumulador y registro de estado) para permitir la reanudación segura o la terminación controlada del proceso.

## 2. Sección Práctica 1: Simulación de Arquitectura de CPU

El Bloque de Control de Proceso (BCP) es la estructura que almacena información de *planificación*, el *estado del proceso* y los *registros del procesador*.

### [CÓDIGO 1.1] Implementación de la estructura de datos

## Estructura del BCP (`BCP.h`)

```cpp
%%writefile BCP.h
#ifndef BCP_H
#define BCP_H
#include <string>
#include <iostream>
#include <iomanip>

#define MODO_USUARIO 0
#define MODO_KERNEL 1

class BCP {
public:
    int pid;
    std::string nombre;
    std::string estado;
    int pc;
    int sp;
    int ac;
    int psw_modo;
    std::string registro_error;

    BCP(int id, std::string nom) : pid(id), nombre(nom), estado("NUEVO"),
                                   pc(0), sp(1000), ac(0), psw_modo(MODO_USUARIO) {}

    void imprimir() const {
        std::cout << "[BCP PID=" << pid << " | Proceso: " << nombre
                  << " | Estado: " << estado << " | PC: 0x" << std::hex
                  << std::setfill('0') << std::setw(4) << pc
                  << " | SP: 0x" << std::setw(4) << sp << std::dec
                  << " | AC: " << ac << " | PSW_Modo: "
                  << (psw_modo == MODO_KERNEL ? "KERNEL" : "USUARIO") << "]\n";
    }
};
#endif
```

In [1]:
%%writefile BCP.h
# ifndef BCP_H
# define BCP_H
# include <string>
# include <iostream>
# include <iomanip>

# define MODO_USUARIO 0
# define MODO_KERNEL 1

class BCP {
public:
    int pid;
    std::string nombre;
    std::string estado;
    int pc;
    int sp;
    int ac;
    int psw_modo;
    std::string registro_error;

    BCP(int id, std::string nom) : pid(id), nombre(nom), estado("NUEVO"),
                                   pc(0), sp(1000), ac(0), psw_modo(MODO_USUARIO) {}

    void imprimir() const {
        std::cout << "[BCP PID=" << pid << " | Proceso: " << nombre
                  << " | Estado: " << estado << " | PC: 0x" << std::hex
                  << std::setfill('0') << std::setw(4) << pc
                  << " | SP: 0x" << std::setw(4) << sp << std::dec
                  << " | AC: " << ac << " | PSW_Modo: "
                  << (psw_modo == MODO_KERNEL ? "KERNEL" : "USUARIO") << "]\n";
    }
};
# endif

Writing BCP.h


El BCP permite que el sistema operativo interrumpa un proceso en ejecución y posteriormente restaure su estado sin perder el hilo de ejecución.

### [CÓDIGO 1.2] Simulador de Hardware (`CPUDualSimulada.h`)

```cpp
%%writefile CPUDualSimulada.h
#ifndef CPU_DUAL_H
#define CPU_DUAL_H
#include "BCP.h"
#include <vector>
#include <thread>
#include <chrono>

struct Instruccion {
    std::string opcode;
    int operando;
    bool privilegiada;
};

class CPUDualSimulada {
private:
    int PC, SP, AC, PSW;
    BCP* proceso_activo;

public:
    CPUDualSimulada() : PC(0), SP(1000), AC(0), PSW(MODO_USUARIO), proceso_activo(nullptr) {}

    void cargar_proceso(BCP* bcp) {
        std::cout << "\n---> [HARDWARE CPU] Cargando contexto del proceso PID " << bcp->pid << "\n";
        proceso_activo = bcp;
        PC = bcp->pc; SP = bcp->sp; AC = bcp->ac; PSW = bcp->psw_modo;
        proceso_activo->estado = "EJECUCION";
    }

    void salvar_contexto() {
        if (proceso_activo) {
            proceso_activo->pc = PC; proceso_activo->sp = SP;
            proceso_activo->ac = AC; proceso_activo->psw_modo = PSW;
        }
    }

    void rutina_tratamiento_interrupcion(std::string tipo, std::string mensaje) {
        std::cout << "\n🚨 [EXCEPCIÓN DE HARDWARE DISPARADA] 🚨\nDetalle: " << mensaje << "\n";
        PSW = MODO_KERNEL;
        salvar_contexto();
        if (tipo == "VIOLACION_DE_PRIVILEGIO" || tipo == "FALLO_ARITMETICO_DIV_ZERO") {
            proceso_activo->estado = "ABORTADO";
            proceso_activo->registro_error = tipo;
        }
        PSW = MODO_USUARIO;
        proceso_activo = nullptr;
    }

    void ejecutar_programa(const std::vector<Instruccion>& programa) {
        for (const auto& inst : programa) {
            if (!proceso_activo || proceso_activo->estado == "ABORTADO") break;
            
            std::cout << "\n[FETCH PC=0x" << std::hex << PC << std::dec << "] Instruccion: " << inst.opcode << "\n";
            PC += 2;
            
            if (inst.privilegiada && PSW == MODO_USUARIO) {
                rutina_tratamiento_interrupcion("VIOLACION_DE_PRIVILEGIO", "Intento de ejecucion privilegiada.");
                break;
            }
            
            if (inst.opcode == "LOAD_AC") AC = inst.operando;
            else if (inst.opcode == "ADD") AC += inst.operando;
            else if (inst.opcode == "DIV") {
                if (inst.operando == 0) {
                    rutina_tratamiento_interrupcion("FALLO_ARITMETICO_DIV_ZERO", "Division por cero.");
                    break;
                }
                AC /= inst.operando;
            }
            std::this_thread::sleep_for(std::chrono::milliseconds(300));
        }
    }
};
#endif
```

In [3]:
%%writefile CPUDualSimulada.h
# ifndef CPU_DUAL_H
# define CPU_DUAL_H
# include "BCP.h"
# include <vector>
# include <thread>
# include <chrono>

struct Instruccion {
    std::string opcode;
    int operando;
    bool privilegiada;
};

class CPUDualSimulada {
private:
    int PC, SP, AC, PSW;
    BCP* proceso_activo;

public:
    CPUDualSimulada() : PC(0), SP(1000), AC(0), PSW(MODO_USUARIO), proceso_activo(nullptr) {}

    void cargar_proceso(BCP* bcp) {
        std::cout << "\n---> [HARDWARE CPU] Cargando contexto del proceso PID " << bcp->pid << "\n";
        proceso_activo = bcp;
        PC = bcp->pc; SP = bcp->sp; AC = bcp->ac; PSW = bcp->psw_modo;
        proceso_activo->estado = "EJECUCION";
    }

    void salvar_contexto() {
        if (proceso_activo) {
            proceso_activo->pc = PC; proceso_activo->sp = SP;
            proceso_activo->ac = AC; proceso_activo->psw_modo = PSW;
        }
    }

    void rutina_tratamiento_interrupcion(std::string tipo, std::string mensaje) {
        std::cout << "\n🚨 [EXCEPCIÓN DE HARDWARE DISPARADA] 🚨\nDetalle: " << mensaje << "\n";
        PSW = MODO_KERNEL;
        salvar_contexto();
        if (tipo == "VIOLACION_DE_PRIVILEGIO" || tipo == "FALLO_ARITMETICO_DIV_ZERO") {
            proceso_activo->estado = "ABORTADO";
            proceso_activo->registro_error = tipo;
        }
        PSW = MODO_USUARIO;
        proceso_activo = nullptr;
    }

    void ejecutar_programa(const std::vector<Instruccion>& programa) {
        for (const auto& inst : programa) {
            if (!proceso_activo || proceso_activo->estado == "ABORTADO") break;

            std::cout << "\n[FETCH PC=0x" << std::hex << PC << std::dec << "] Instruccion: " << inst.opcode << "\n";
            PC += 2;

            if (inst.privilegiada && PSW == MODO_USUARIO) {
                rutina_tratamiento_interrupcion("VIOLACION_DE_PRIVILEGIO", "Intento de ejecucion privilegiada.");
                break;
            }

            if (inst.opcode == "LOAD_AC") AC = inst.operando;
            else if (inst.opcode == "ADD") AC += inst.operando;
            else if (inst.opcode == "DIV") {
                if (inst.operando == 0) {
                    rutina_tratamiento_interrupcion("FALLO_ARITMETICO_DIV_ZERO", "Division por cero.");
                    break;
                }
                AC /= inst.operando;
            }
            std::this_thread::sleep_for(std::chrono::milliseconds(300));
        }
    }
};
# endif

Overwriting CPUDualSimulada.h


### [CÓDIGO 1.3] Punto de Entrada (`main.cpp`)

```cpp
%%writefile main.cpp
#include "CPUDualSimulada.h"
#include <iostream>

int main() {
    CPUDualSimulada cpu;

    std::cout << "\nESCENARIO 1: VIOLACION DE PRIVILEGIO\n";
    BCP bcp1(101, "NavegadorWeb.exe");
    cpu.cargar_proceso(&bcp1);
    std::vector<Instruccion> prog1 = {
        {"LOAD_AC", 50, false}, {"ADD", 25, false},
        {"IN_PORT_IO", 0x3F8, true}, {"ADD", 10, false}
    };
    cpu.ejecutar_programa(prog1);
    bcp1.imprimir();

    std::cout << "\nESCENARIO 2: FALLO ARITMETICO\n";
    BCP bcp2(102, "Calculadora.exe");
    cpu.cargar_proceso(&bcp2);
    std::vector<Instruccion> prog2 = {
        {"LOAD_AC", 100, false}, {"DIV", 0, false}, {"ADD", 5, false}
    };
    cpu.ejecutar_programa(prog2);
    bcp2.imprimir();

    return 0;
}
```

In [4]:
%%writefile main.cpp
# include "CPUDualSimulada.h"
# include <iostream>

int main() {
    CPUDualSimulada cpu;

    std::cout << "\nESCENARIO 1: VIOLACION DE PRIVILEGIO\n";
    BCP bcp1(101, "NavegadorWeb.exe");
    cpu.cargar_proceso(&bcp1);
    std::vector<Instruccion> prog1 = {
        {"LOAD_AC", 50, false}, {"ADD", 25, false},
        {"IN_PORT_IO", 0x3F8, true}, {"ADD", 10, false}
    };
    cpu.ejecutar_programa(prog1);
    bcp1.imprimir();

    std::cout << "\nESCENARIO 2: FALLO ARITMETICO\n";
    BCP bcp2(102, "Calculadora.exe");
    cpu.cargar_proceso(&bcp2);
    std::vector<Instruccion> prog2 = {
        {"LOAD_AC", 100, false}, {"DIV", 0, false}, {"ADD", 5, false}
    };
    cpu.ejecutar_programa(prog2);
    bcp2.imprimir();

    return 0;
}

Writing main.cpp


### Compilación y Ejecución

Para ejecutar esta simulación, utilizaremos el compilador `g++` (el compilador de GNU para C++) para traducir el código fuente a un ejecutable.

```cpp
!g++ main.cpp -o simulador_cpu -std=c++11
!./simulador_cpu
```

In [5]:
!g++ main.cpp -o simulador_cpu -std=c++11
!./simulador_cpu


ESCENARIO 1: VIOLACION DE PRIVILEGIO

---> [HARDWARE CPU] Cargando contexto del proceso PID 101

[FETCH PC=0x0] Instruccion: LOAD_AC

[FETCH PC=0x2] Instruccion: ADD

[FETCH PC=0x4] Instruccion: IN_PORT_IO

🚨 [EXCEPCIÓN DE HARDWARE DISPARADA] 🚨
Detalle: Intento de ejecucion privilegiada.
[BCP PID=101 | Proceso: NavegadorWeb.exe | Estado: ABORTADO | PC: 0x0006 | SP: 0x03e8 | AC: 75 | PSW_Modo: KERNEL]

ESCENARIO 2: FALLO ARITMETICO

---> [HARDWARE CPU] Cargando contexto del proceso PID 102

[FETCH PC=0x0] Instruccion: LOAD_AC

[FETCH PC=0x2] Instruccion: DIV

🚨 [EXCEPCIÓN DE HARDWARE DISPARADA] 🚨
Detalle: Division por cero.
[BCP PID=102 | Proceso: Calculadora.exe | Estado: ABORTADO | PC: 0x0004 | SP: 0x03e8 | AC: 100 | PSW_Modo: KERNEL]


## 3. Sección de Autoevaluación: Arquitectura de CPU, Modo Dual y BCP

In [6]:
#@title Responder la Pregunta 1 { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Definir la pregunta y las opciones
pregunta = widgets.HTML(value="<h3><b>Pregunta 1:</b> ¿Durante la ejecución de la instrucción IN_PORT_IO en Modo Usuario, el hardware de la CPU dispara un <strong>TRAP</strong> de violación de privilegio.<br>¿Cuál es el orden exacto de las operaciones que realiza el hardware antes de ceder el control a la rutina del kernel?</h3>")

opciones = widgets.RadioButtons(
    options=[
        'A) Cambia la PSW a Modo Usuario, incrementa el PC, y guarda el BCP en el disco.',
        'B) Cancela la instrucción, salva PC/PSW actuales en la pila o BCP, eleva la PSW a Modo Kernel y salta a la dirección de la ISR.',
        'C) Aborta el sistema operativo, limpia la memoria RAM y reinicia la CPU en Modo Kernel.',
        'D) Ejecuta la instrucción en Modo Usuario y luego notifica al proceso.'
    ],
    value=None, # Inicia sin ninguna opción seleccionada
    description='',
    disabled=False,
    layout=widgets.Layout(width='100%')
)

# 2. Crear el botón de verificación y el contenedor de salida
boton_verificar = widgets.Button(description="Verificar Respuesta", button_style='info')
salida_retroalimentacion = widgets.Output()

# 3. Definir la lógica de la retroalimentación
def evaluar_respuesta(b):
    with salida_retroalimentacion:
        clear_output() # Limpia la retroalimentación anterior

        if opciones.value is None:
            print("⚠️ Por favor, selecciona una opción antes de verificar.")
            return

        # Validar la respuesta correcta
        if opciones.value == 'B) Cancela la instrucción, salva PC/PSW actuales en la pila o BCP, eleva la PSW a Modo Kernel y salta a la dirección de la ISR.':
            print("✅ ¡Correcto! Al detectarse la instrucción privilegiada en nivel no privilegiado, la CPU detiene el ciclo de instrucción, preserva el estado de control actual (PC/PSW) y eleva de forma forzosa el bit de privilegio en la PSW a Modo Kernel antes de transferir el control a la rutina de interrupción")
        else:
            print("❌ Incorrecto.")

# Conectar el botón con la función de evaluación
boton_verificar.on_click(evaluar_respuesta)

# 4. Mostrar todos los elementos en pantalla
display(pregunta, opciones, boton_verificar, salida_retroalimentacion)

HTML(value='<h3><b>Pregunta 1:</b> ¿Durante la ejecución de la instrucción IN_PORT_IO en Modo Usuario, el hard…

RadioButtons(layout=Layout(width='100%'), options=('A) Cambia la PSW a Modo Usuario, incrementa el PC, y guard…

Button(button_style='info', description='Verificar Respuesta', style=ButtonStyle())

Output()

In [7]:
#@title Responder la Pregunta 2 { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Definir la pregunta y las opciones
pregunta = widgets.HTML(value="<h3><b>Pregunta 2:</b> En el código C++ del simulador de la Práctica 1, si un proceso comete una división por cero (DIV 0), el BCP del proceso termina en estado ABORTADO.<br> Si quisiéramos que el programa continuara su ejecución ignorando el error aritmético (poniendo el acumulador AC = 0),<br>¿qué modificación debería hacerse en la rutina de tratamiento de interrupción (rutina_tratamiento_interrupcion)?</h3>")

opciones = widgets.RadioButtons(
    options=[
        'A) Cambiar el puntero de pila `SP = 0` y reiniciar la CPU.',
        'B) Asignar `PC = 0` para que el programa empiece desde la primera línea.',
        'C) Eliminar el BCP de la memoria.',
        'D) Modificar la PSW a `MODO_USUARIO`, asignar `AC = 0`, mantener el estado como `EJECUCION` y permitir que la rutina devuelva el control.'
    ],
    value=None, # Inicia sin ninguna opción seleccionada
    description='',
    disabled=False,
    layout=widgets.Layout(width='100%')
)

# 2. Crear el botón de verificación y el contenedor de salida
boton_verificar = widgets.Button(description="Verificar Respuesta", button_style='info')
salida_retroalimentacion = widgets.Output()

# 3. Definir la lógica de la retroalimentación
def evaluar_respuesta(b):
    with salida_retroalimentacion:
        clear_output() # Limpia la retroalimentación anterior

        if opciones.value is None:
            print("⚠️ Por favor, selecciona una opción antes de verificar.")
            return

        # Validar la respuesta correcta
        if opciones.value == 'D) Modificar la PSW a `MODO_USUARIO`, asignar `AC = 0`, mantener el estado como `EJECUCION` y permitir que la rutina devuelva el control.':
            print("✅ ¡Correcto! Para recuperarse de un fallo aritmético suave, el kernel corrige el registro de trabajo (AC = 0), mantiene el estado del proceso activo y restaura el nivel de ejecución a Modo Usuario para continuar el flujo de instrucciones")
        else:
            print("❌ Incorrecto.")

# Conectar el botón con la función de evaluación
boton_verificar.on_click(evaluar_respuesta)

# 4. Mostrar todos los elementos en pantalla
display(pregunta, opciones, boton_verificar, salida_retroalimentacion)

HTML(value='<h3><b>Pregunta 2:</b> En el código C++ del simulador de la Práctica 1, si un proceso comete una d…

RadioButtons(layout=Layout(width='100%'), options=('A) Cambiar el puntero de pila `SP = 0` y reiniciar la CPU.…

Button(button_style='info', description='Verificar Respuesta', style=ButtonStyle())

Output()